# High-Effort Optimization Playbook

The HIGH-effort tier is for when the lower tiers aren't enough — compound-AI patterns that add real complexity, earned on evidence. Reach for them only after LOW and MEDIUM are applied, measured, and still short of target.

| # | Lever | Mode |
|---|---|---|
| 1 | Harness Engineering (Lever 13) | Concept — the practice that ties the other levers together |
| 2 | Sub-Agent Delegation (Lever 14) | Hands-on — Claude Agent SDK subagents (Opus lead + Haiku worker) |
| 3 | GEPA / DSPy (Lever 15) | Theory + conceptual code (a real `compile()` is too long for a live slot) |
| 4 | Tool Search via MCP Gateway (Lever 16) | Concept + the search call (full deploy lives in the Developer Journey) |

## Setup

In [ ]:
import os, time, json
import boto3

REGION = "us-east-1"
RUNTIME = boto3.client("bedrock-runtime", region_name=REGION)

OPUS = "global.anthropic.claude-opus-4-8"
SONNET = "global.anthropic.claude-sonnet-4-6"
HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"


---

## 1. Harness Engineering — Lever 13

The naive view is "the model is the agent." The current view: the model is one component; the **harness** is the deterministic scaffolding around it — tools, context curation, the control loop, retries, evals, guardrails. Engineer it and the model's average behavior tracks much closer to its peak; don't, and you live in failure-mode whack-a-mole.

> "Every time the agent makes a mistake, change the system so it cannot make that mistake again." — Mitchell Hashimoto (2026)

An agent is just **"an LLM using tools in a loop"**: call the model, parse output, run tools, append results, check a stop condition, repeat. The loop is where cost and latency live — step one re-sends the *entire growing context* every pass, and each pass is a full round-trip. Two numbers dominate your bill and p95 latency:

- **Tokens per turn** — lean context, because you pay for it *every* turn.
- **Turns per task** — fewer round-trips to the answer.

<div class="alert alert-block alert-info">

**The harness loop, costed.** Input bloat is multiplied by turn count; each turn adds a round-trip. Trimming a 3,000-token system prompt to 800 saves those 2,200 tokens on turn 1 *and* turn 8; consolidating two tool calls into one removes a whole round-trip. OpenAI's rule of thumb: cutting 50% of *output* tokens can cut ~50% of latency, while cutting input yields only ~1–5% — output brevity is the latency lever, input is the cost/cache lever. Small per-turn savings compound.

</div>

**Don't reach for an agent you don't need.** A fixed *workflow* (code-defined path) is cheaper and lower-latency than a model-directed agent whenever the path is predictable. Climb the ladder — single call → chaining → routing → parallelization → orchestrator-workers → autonomous agent — only as far as the task forces you.

### What the harness includes

- **Context curation** — decide what enters the window each turn. Static prompts cached, dynamic context retrieved just-in-time, conversation summarized at boundaries. Every token is a deliberate choice.
- **Tool design** — a tool description is a prompt the model reads to decide whether to call it. Name the verb, list inputs in order, state the output format. Keep a minimal, non-overlapping set (every description is re-sent every turn), consolidate multi-step calls to cut round-trips, and compress tool results to high-signal fields.
- **The control loop + retries** — deterministic code that drives call → parse → act → check-stop, retries transient failures, and enforces a turn budget so a runaway loop can't quietly 10× your bill.
- **Evals** — pair every tool and prompt with a test on realistic input. When a description drifts or a model upgrades, the eval catches the regression before production does.
- **Deterministic guardrails** — things that must never happen (a refund over $X without approval, an unscoped SQL query) are enforced by code, not prompt instructions. The model cooperates; code enforces.

### A production-readiness checklist

Before declaring an agent production-ready, audit each of these — it's the concrete form of "is my harness real?":

- [ ] **Tools** — every tool's description names the verb, the inputs, and the output format
- [ ] **Tool evals** — every tool has ≥1 eval exercising it on realistic input
- [ ] **Context strategy** — explicit decision on what's cached, retrieved, summarized, per-request
- [ ] **Failure tracing** — every production failure lands in Langfuse and gets classified
- [ ] **Eval-as-code** — failures become eval cases; evals run on every prompt change; a CI gate fails if the score drops
- [ ] **Code-level guardrails** — anything irreversible (refunds, deletes, external sends) is rate-limited or human-approved by code, not prompt
- [ ] **Versioning** — prompts versioned, models pinned, past behavior reconstructable

Tick all seven and you have a harness. Tick fewer than three and you should fix that **before** reaching for HIGH-tier patterns like sub-agents — they amplify a weak harness (a badly described tool just makes three agents wrong, faster).

### What's covered where

The harness is a practice, not an API, so you build each component in the lab where it fits — there's no standalone harness notebook. This is the map:

| Harness component | Where you practice it |
|---|---|
| **Context curation** | Caching ([`01-low-effort.ipynb`](./01-low-effort.ipynb)), RAG + conversation/memory ([`02-medium-effort.ipynb`](./02-medium-effort.ipynb)) |
| **Tool design + evals** | Developer Journey — [`06-agentcore-gateway.ipynb`](../03-developer-journey/06-agentcore-gateway.ipynb), [`07-evaluations.ipynb`](../03-developer-journey/07-evaluations.ipynb) |
| **Deterministic guardrails** | Guardrails (Lever 08, [`02-medium-effort.ipynb`](./02-medium-effort.ipynb)) + your own code checks |
| **Control loop, retries, feedback** | Production Prompt Lifecycle — [`04-deep-dive-topics/02-production-prompt-lifecycle.ipynb`](../04-deep-dive-topics/02-production-prompt-lifecycle.ipynb) (CI/CD eval gates) |

<div class="alert alert-block alert-warning">

**The harness rots.** Tool descriptions drift, evals decay, prompts go stale on model upgrades. Budget recurring engineering time, or quality regresses silently while cost creeps up.

</div>

---

## 2. Sub-Agent Delegation — Lever 14

A single agent that does everything carries every tool result, every search hit, every intermediate step in one context window. That window grows fast and gets re-sent every turn — by turn 50 the agent spends most of its budget re-reading its own past work, getting slower and worse at the task.

**Sub-agent delegation** breaks this with an **orchestrator-worker** split: a **lead** plans and answers; a **worker** does the token-heavy subtask (search, extract, summarize) in its *own* isolated context and returns only a compact result. The worker's raw material — tens of thousands of tokens of search hits, retries, partial drafts — never enters the lead's window.

The cost lever is the model choice: workers do bounded, well-specified jobs, so route them to a **cheap** model and keep the **strong** model on the lead, where it sees only the question and the compressed answer.

<div class="alert alert-block alert-info">

**Mind the orchestration overhead.** Every delegation is a separate Bedrock invocation. The win is the lead's context savings — make sure it exceeds that overhead. Sub-agents are no substitute for a good harness (Lever 13): a worker with bad tool descriptions is just bad, faster.

</div>

We use the **Claude Agent SDK**: declare named subagents in `ClaudeAgentOptions(agents={...})` and the lead delegates automatically, only the result returning. On Bedrock, set `CLAUDE_CODE_USE_BEDROCK=1` and pick models with `ANTHROPIC_MODEL` (lead) and `ANTHROPIC_SMALL_FAST_MODEL`; an `AgentDefinition` can pin its own `model` alias. Below, the lead runs on Opus and `research_worker` on Haiku.

In [ ]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, AgentDefinition, AssistantMessage, TextBlock

# Drive the SDK against Bedrock. CLAUDE_CODE_USE_BEDROCK=1 selects the Bedrock
# backend; the two model env vars pick the lead and the default small model.
SDK_ENV = {
    "CLAUDE_CODE_USE_BEDROCK": "1",
    "ANTHROPIC_MODEL": OPUS,                # the LEAD model — sees only the question + summary
    "ANTHROPIC_SMALL_FAST_MODEL": HAIKU,    # default small/fast model
    "AWS_REGION": REGION,
}

WORKER_PROMPT = (
    "You are a research worker. You receive one research question and answer it "
    "from your own knowledge as a 5-bullet summary. Return ONLY the final "
    "summary — no preamble, no raw sources, no intermediate reasoning."
)

options = ClaudeAgentOptions(
    system_prompt=(
        "You are a lead agent. For anything that needs researching or "
        "cross-referencing, delegate to the research_worker subagent and build "
        "your answer from its summary. Answer simple questions directly."
    ),
    agents={
        # A named subagent. `description` tells the lead WHEN to delegate;
        # `prompt` is the worker's system prompt; `model` pins it to the cheap tier.
        # tools=[] keeps the worker a pure reasoner — no file/shell tools, so it
        # answers from knowledge instead of hunting the workspace.
        "research_worker": AgentDefinition(
            description="Answer a research question from knowledge and return a compressed 5-bullet summary.",
            prompt=WORKER_PROMPT,
            model="haiku",   # worker runs on Haiku; the lead stays on Opus
            tools=[],        # no tools — isolated, pure-reasoning worker
        ),
    },
    env=SDK_ENV,
    setting_sources=[],                 # SDK-only: don't load local filesystem settings
    permission_mode="bypassPermissions",
)

In [ ]:
USER_QUERY = (
    "What are the most effective levers for reducing LLM inference cost and "
    "latency on Amazon Bedrock? Delegate the research to research_worker, then "
    "give me a brief synthesized answer."
)


async def run_lead(prompt: str):
    final_text, delegated = [], []
    async for message in query(prompt=prompt, options=options):
        for block in getattr(message, "content", []) or []:
            # A ToolUseBlock named "Agent" is the lead delegating to a subagent.
            if type(block).__name__ == "ToolUseBlock" and getattr(block, "name", "") == "Agent":
                delegated.append((block.input or {}).get("subagent_type", "subagent"))
            # The lead's own text (its synthesized answer).
            if isinstance(message, AssistantMessage) and isinstance(block, TextBlock):
                final_text.append(block.text)
    return "\n".join(t for t in final_text if t.strip()), delegated


# query() is async; in a notebook cell you can await it directly.
answer, delegations = await run_lead(USER_QUERY)
print(answer)
print("\nDelegated to subagent:", delegations or "(none — lead answered directly)")

**What you'll see**: the printed answer is the lead's synthesis; the "Delegated to subagent" line confirms the work was handed to `research_worker`. The lead's context holds only the question, the worker's summary, and the final answer — not the material the worker read through.

<div class="alert alert-block alert-success">

**Alternative: the Strands Agents SDK.** If your Developer-Journey agent is already on [Strands](https://strandsagents.com/), the same orchestrator-worker pattern is **agents-as-tools** — wrap a worker `Agent` in an `@tool` and hand it to the lead. The cost lever is identical: a strong lead (Opus) coordinating a cheaper worker (Haiku), the worker's token-heavy context isolated. The cell below runs it.

</div>

In [ ]:
# Same orchestrator-worker pattern in Strands: a worker Agent wrapped as an @tool.
from strands import Agent, tool
from strands.models import BedrockModel


@tool
def research_worker(query: str) -> str:
    """Research a question and return a compressed 5-bullet summary."""
    worker = Agent(
        model=BedrockModel(model_id=HAIKU, region_name=REGION),   # cheap worker
        system_prompt="Answer from your knowledge as a 5-bullet summary. No raw sources.",
        callback_handler=None,                                    # silence the worker's stream
    )
    return str(worker(query))


strands_lead = Agent(
    model=BedrockModel(model_id=OPUS, region_name=REGION),        # strong lead
    system_prompt="Delegate research to the research_worker tool; answer simple questions directly.",
    tools=[research_worker],
    callback_handler=None,
)

result = strands_lead(
    "What are the most effective levers for cutting LLM inference cost and latency on "
    "Amazon Bedrock? Delegate the research to research_worker, then synthesize briefly."
)
print(result.message["content"][0]["text"])

# Tool calls the lead made (delegations to the worker show up in its message history).
delegations = [
    block["toolUse"]["name"]
    for msg in strands_lead.messages
    for block in msg.get("content", [])
    if isinstance(block, dict) and "toolUse" in block
]
print("\nDelegated tool calls:", delegations or "(none — lead answered directly)")

---

## 3. GEPA / DSPy — Lever 15

**Theory + conceptual code.** A `compile()` run needs tens to hundreds of pipeline evaluations — minutes to hours — plus a training set and a feedback function, so it doesn't fit a live slot. This section shows the structure to run on your own task later.

GEPA (**Ge**netic-**Pa**reto) treats your prompt pipeline as a *program* and lets an eval-driven optimizer rewrite it:

1. **Run** the current prompts on a training minibatch; capture full trajectories (reasoning, tool calls, outputs, scores).
2. **Reflect** — a strong LM reads those traces *in natural language*, diagnoses failures, proposes prompt edits.
3. **Pareto-select** — keep candidates that win on *at least one* example (not just best-on-average), preserving diversity instead of collapsing onto one brittle prompt.
4. **Loop** until validation plateaus, then ship. At runtime the compiled prompt is just one inference call.

Language-level reflection learns faster than scalar reward — the paper reports GEPA beating the RL baseline GRPO and the earlier MIPRO, with far fewer rollouts. The cost/latency payoff: optimization is a **one-time offline** cost, but a cheaper model can end up hitting the quality bar of an expensive one.

<div class="alert alert-block alert-warning">

**The managed counterpart is Bedrock APO (Lever 05)** — one API call, single-prompt, console/API, first-class model migration. Reach for GEPA/DSPy when you need open-source control over the loop, a custom feedback function, or multi-stage pipelines. Break-even: if you'd hand-iterate prompts for a day and the pipeline runs ≥10K times/month, the offline optimization wins. **The feedback function is the hard part** — a weak metric optimizes for the wrong thing, so invest there more than in the prompt.

</div>

### The DSPy program

Declare *what* the task is (a typed `Signature`) and *how* to solve it (a `Module`). There's no prompt string — DSPy builds it, and GEPA later rewrites it.

<div class="alert alert-block alert-info">

The cells below are **illustrative / conceptual** — they show the API shape, not a runnable compile. To run for real: `pip install dspy`, supply a real `trainset`/`valset`, and budget for many model calls.

</div>

In [ ]:
# ILLUSTRATIVE / CONCEPTUAL — shows DSPy structure, not a runnable compile.
import dspy

# Point DSPy at Bedrock. DSPy uses LiteLLM under the hood, so model IDs take a
# "bedrock/" prefix. The task model is the cheap one we want to make reliable.
dspy.configure(
    lm=dspy.LM("bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0"),
)

# A Signature declares the task's typed inputs/outputs in plain English field names.
class ClassifyTicket(dspy.Signature):
    """Classify a support ticket into a category and urgency."""
    ticket: str = dspy.InputField()
    category: str = dspy.OutputField(desc="billing | technical | account | other")
    urgency: str = dspy.OutputField(desc="low | medium | high")

# A Module composes Signatures into a program. ChainOfThought adds a reasoning step.
class TicketRouter(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classify = dspy.ChainOfThought(ClassifyTicket)

    def forward(self, ticket: str) -> dspy.Prediction:
        return self.classify(ticket=ticket)

program = TicketRouter()

### The metric — where the learning signal comes from

GEPA's metric returns *more than a number*. Return `dspy.Prediction(score, feedback)` and the `feedback` string is fed to the reflection model — it tells GEPA **why** a prediction was wrong, the richer signal that beats scalar RL rewards.

In [ ]:
# ILLUSTRATIVE / CONCEPTUAL — metric returns a score AND natural-language feedback.
# GEPA calls it with these args; pred_name/pred_trace target a specific predictor.
def metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    score = 0.0
    notes = []

    if pred.category == gold.category:
        score += 0.5
    else:
        notes.append(f"category wrong: said '{pred.category}', expected '{gold.category}'")

    if pred.urgency == gold.urgency:
        score += 0.5
    else:
        notes.append(f"urgency wrong: said '{pred.urgency}', expected '{gold.urgency}'")

    feedback = "Correct." if score == 1.0 else "Fix these: " + "; ".join(notes)
    # Returning Prediction(score=..., feedback=...) gives GEPA the reflective signal.
    return dspy.Prediction(score=score, feedback=feedback)

### Optimize: `GEPA.compile()`

Same shape as every DSPy optimizer: construct with the metric, then `compile(student, trainset=, valset=)`. GEPA adds a stronger `reflection_lm` (it reads trajectories and rewrites instructions) and an `auto` budget. The optimizer is constructed below; the actual `compile()` is **gated behind `RUN_GEPA=1`** (with a real `trainset`/`valset`) so a Run-All doesn't kick off a multi-minute optimization.

In [ ]:
# ILLUSTRATIVE / CONCEPTUAL — a real run makes many model calls over a real dataset,
# so this is gated off by default (Run-All won't trigger a long compile). To run it for
# real: supply `trainset`/`valset` as lists of dspy.Example and set RUN_GEPA=1.
import os
from dspy import GEPA

optimizer = GEPA(
    metric=metric,
    auto="light",  # budget preset: "light" | "medium" | "heavy"
    # The reflection model should be strong — it does the prompt rewriting.
    reflection_lm=dspy.LM("bedrock/global.anthropic.claude-opus-4-8"),
    candidate_selection_strategy="pareto",  # keep the Pareto frontier (the default)
    track_stats=True,
)

if os.getenv("RUN_GEPA") == "1":
    # compile() returns a NEW program whose prompts GEPA has rewritten.
    optimized = optimizer.compile(
        program,
        trainset=trainset,   # list[dspy.Example] — you supply these
        valset=valset,       # list[dspy.Example] — scored for Pareto selection
    )
    optimized.save("ticket_router.optimized.json")   # the evolved prompt, ready to deploy
    # optimized.detailed_results  # per-candidate scores + Pareto frontier (track_stats=True)
    print("GEPA compile complete — optimized prompt saved.")
else:
    print("GEPA optimizer constructed. Set RUN_GEPA=1 (with a real trainset/valset) to compile.")
    print("A real compile runs tens-to-hundreds of evaluations — minutes to hours.")

<div class="alert alert-block alert-success">

**Take-away.** GEPA turns prompt tuning into a measured search: define the task, write a metric that explains *why* an answer is wrong, and let reflection evolve the prompt. The reward is a cheaper model that hits the quality bar of a more expensive one — directly on this playbook's cost/latency target.

</div>

**References** — Agrawal et al., *GEPA* (arXiv [2507.19457](https://arxiv.org/abs/2507.19457)) · [DSPy framework](https://dspy.ai) · [GEPA optimizer docs](https://dspy.ai/api/optimizers/GEPA/overview/)

---

## 4. Tool Search via MCP Gateway — Lever 16

A production agent accumulates tools — order lookup, refunds, KB search, ticketing, escalation. Send all 30+ definitions every turn and you inflate input tokens and time-to-first-token, even though most queries need 2-3. Tool search inverts it: index every tool's metadata once, retrieve only the top-K relevant tools per query, pass *just those* to the model.

**AgentCore Gateway** is the managed version. Enable semantic search at `CreateGateway` (`searchType="SEMANTIC"` — **can't be added later**); each `CreateTarget` then auto-embeds its tools' names, descriptions, and input/output schemas into a serverless vector store. The gateway exposes one built-in tool, `x_amz_bedrock_agentcore_search` — call it instead of MCP's `tools/list` to get ranked matches back in under a second, even across hundreds of tools.

<div class="alert alert-block alert-warning">

**The key trade-off: tool search breaks tool-definition caching.** Tool schemas are normally the most stable part of your prompt and cache at position 0 (Lever 04). A *different* tool set per query changes that prefix, so you lose the cache hit. The math only works once the catalog is large enough (**15+ tools**) that per-request token savings outweigh the lost cache discount. Below that, static + cached beats dynamic + uncached.

</div>

<div class="alert alert-block alert-info">

**The full deploy lives in the Developer Journey** — [`03-developer-journey/06-agentcore-gateway.ipynb`](../03-developer-journey/06-agentcore-gateway.ipynb) builds a gateway end-to-end with Cognito auth and Lambda targets. Here we show the **create** call (gated — needs that infra) and the **search** call (the highlight, runs against any gateway you have).

</div>

In [ ]:
import os
import time
import boto3

REGION = os.getenv("AWS_REGION", "us-east-1")

# --- Pre-provisioned infra, read from environment (set from your CFN outputs) ---
# Inbound OAuth (Cognito) for the gateway's CUSTOM_JWT authorizer:
GW_CLIENT_ID   = os.getenv("GW_CLIENT_ID", "")        # Cognito app client id (M2M)
DISCOVERY_URL  = os.getenv("GW_DISCOVERY_URL", "")    # .../.well-known/openid-configuration
# IAM role the gateway assumes to invoke target Lambdas (needs lambda:InvokeFunction):
GW_ROLE_ARN    = os.getenv("GW_ROLE_ARN", "")
# A Lambda to back the first target:
TARGET_LAMBDA_ARN = os.getenv("TARGET_LAMBDA_ARN", "")

GATEWAY_NAME = f"playbook-toolsearch-{int(time.time())}"

# bedrock-agentcore-control is the gateway management plane.
gw_admin = boto3.client("bedrock-agentcore-control", region_name=REGION)

print("Region:", REGION)
print("Deploy gated by RUN_GATEWAY_DEPLOY =", os.getenv("RUN_GATEWAY_DEPLOY", "0"))

### Create a gateway with semantic search, then register a Lambda target

The `searchType="SEMANTIC"` flag on `protocolConfiguration.mcp` enables the built-in search tool. Each target's `toolSchema.inlinePayload` is the list of tool definitions (name, description, input schema) the gateway indexes for retrieval. *Gated behind `RUN_GATEWAY_DEPLOY=1` — needs the Cognito + Lambda + IAM infra above.*

In [ ]:
# Gated: only runs when RUN_GATEWAY_DEPLOY=1 AND the infra identifiers are set.
GATEWAY_ID = os.getenv("GATEWAY_ID", "")
GATEWAY_URL = os.getenv("GATEWAY_URL", "")

_have_infra = all([GW_CLIENT_ID, DISCOVERY_URL, GW_ROLE_ARN, TARGET_LAMBDA_ARN])

if os.getenv("RUN_GATEWAY_DEPLOY") == "1" and _have_infra:
    # 1) Create the gateway WITH semantic search enabled.
    resp = gw_admin.create_gateway(
        name=GATEWAY_NAME,
        roleArn=GW_ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "allowedClients": [GW_CLIENT_ID],
                "discoveryUrl": DISCOVERY_URL,
            }
        },
        protocolConfiguration={
            "mcp": {
                "searchType": "SEMANTIC",          # <-- enables x_amz_bedrock_agentcore_search
                "supportedVersions": ["2025-11-25"],
            }
        },
        exceptionLevel="DEBUG",
    )
    GATEWAY_ID = resp["gatewayId"]
    GATEWAY_URL = resp["gatewayUrl"]
    print("Gateway:", GATEWAY_ID)

    # Wait until READY before adding targets.
    while True:
        status = gw_admin.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
        print("  status:", status)
        if status in ("READY", "FAILED", "CREATE_FAILED"):
            break
        time.sleep(10)

    # 2) Register a Lambda-backed target. The toolSchema.inlinePayload is the
    #    list of tool definitions the gateway will index for semantic search.
    tool_schema = [
        {
            "name": "add_numbers",
            "description": "Add two numbers and return the sum.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                },
                "required": ["a", "b"],
            },
        }
    ]
    tgt = gw_admin.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name="CalcTools",
        description="Calculation tools",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": TARGET_LAMBDA_ARN,
                    "toolSchema": {"inlinePayload": tool_schema},
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    print("Target:", tgt["targetId"])
    print("GATEWAY_URL =", GATEWAY_URL)
else:
    print("Skipping create (set RUN_GATEWAY_DEPLOY=1 and the GW_* / TARGET_LAMBDA_ARN env vars).")
    print("Using existing GATEWAY_URL =", GATEWAY_URL or "(none set)")

### Call the built-in search tool — the highlight

`x_amz_bedrock_agentcore_search` is exposed automatically when semantic search is on. Calling it is one MCP `tools/call` over HTTP: a JSON-RPC POST to the gateway URL with a `Bearer` token. The ranked top-K tools come back in `result.structuredContent.tools` — typically under a second regardless of catalog size. Get the token from your Cognito app client with the `client_credentials` grant.

In [ ]:
import requests

# Obtain an inbound OAuth token (client_credentials). If you already minted one,
# set GW_ACCESS_TOKEN and this call is skipped.
def get_token(token_endpoint, client_id, client_secret, scope):
    r = requests.post(
        token_endpoint,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": scope,
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["access_token"]

GW_ACCESS_TOKEN = os.getenv("GW_ACCESS_TOKEN", "")
if not GW_ACCESS_TOKEN and os.getenv("GW_TOKEN_ENDPOINT"):
    GW_ACCESS_TOKEN = get_token(
        os.environ["GW_TOKEN_ENDPOINT"],
        os.environ["GW_CLIENT_ID"],
        os.environ["GW_CLIENT_SECRET"],
        os.environ.get("GW_SCOPE", ""),
    )

print("Token set:", bool(GW_ACCESS_TOKEN))

In [ ]:
if GATEWAY_URL and GW_ACCESS_TOKEN:
    QUERY = "tools for adding two numbers"

    search_request = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": "x_amz_bedrock_agentcore_search",
            "arguments": {"query": QUERY},
        },
    }
    start = time.time()
    resp = requests.post(
        GATEWAY_URL,
        json=search_request,
        headers={
            "Authorization": f"Bearer {GW_ACCESS_TOKEN}",
            "Content-Type": "application/json",
            "MCP-Protocol-Version": "2025-11-25",
        },
        timeout=30,
    )
    resp.raise_for_status()
    result = resp.json()

    # Ranked top-K tools land in result.structuredContent.tools
    tools_found = result.get("result", {}).get("structuredContent", {}).get("tools", [])
    print(f"Query: {QUERY!r}")
    print(f"Search completed in {time.time() - start:.2f}s")
    print(f"Tools returned: {len(tools_found)}")
    for t in tools_found[:5]:
        print(f"  - {t['name']}: {t.get('description', '')[:60]}")
else:
    print("Set GATEWAY_URL and GW_ACCESS_TOKEN to run the search.")

**What you'll see**: the search returns only the top-K relevant tools (in `result.structuredContent.tools`, ranked best-first) — typically under a second no matter how many tools the gateway hosts. That handful is what reaches the model, so per-turn tool-definition tokens stay flat as the catalog grows from 5 to 300+. At small catalogs it's pure overhead; the win compounds with scale, and the gateway also centralises auth across all tools (one OAuth boundary instead of per-tool credentials).

### (Optional) Hand the searched tools to a Strands agent

Instead of building the agent with all 300+ gateway tools, feed it only the top matches from the search above — the prompt carries a handful of tool schemas, not the whole catalogue.

> Requires `strands-agents` and `mcp`. Skips cleanly if either is missing or no tools were found.

In [ ]:
try:
    import httpx
    from mcp.client.streamable_http import streamable_http_client
    from mcp.types import Tool as MCPTool
    from strands import Agent
    from strands.models import BedrockModel
    from strands.tools.mcp.mcp_client import MCPAgentTool, MCPClient

    _ready = bool(GATEWAY_URL and GW_ACCESS_TOKEN and tools_found)
except ImportError:
    _ready = False
    print("strands-agents / mcp not installed - skipping. (pip install strands-agents mcp)")

if _ready:
    # The current streamable_http_client takes a pre-configured httpx client (not headers=),
    # so the Bearer token goes on the client's default headers.
    client = MCPClient(
        lambda: streamable_http_client(
            GATEWAY_URL,
            http_client=httpx.AsyncClient(
                headers={"Authorization": f"Bearer {GW_ACCESS_TOKEN}"}
            ),
        )
    )
    model = BedrockModel(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
        temperature=0.7,
    )

    with client:
        # Wrap only the searched (top-K) tools for the agent.
        strands_tools = [
            MCPAgentTool(
                MCPTool(
                    name=t["name"],
                    description=t.get("description", ""),
                    inputSchema=t["inputSchema"],
                ),
                client,
            )
            for t in tools_found[:3]
        ]
        print("Agent tools:", [t.tool_name for t in strands_tools])

        agent = Agent(model=model, tools=strands_tools)
        result = agent("add 100 plus 50")
        print("\nAgent response:", result.message["content"][0]["text"])
else:
    print("Skipping Strands agent (need gateway URL, token, and a non-empty search result).")

### Cleanup

Delete the target and gateway you created (skips if you used a pre-existing one).

In [ ]:
if os.getenv("RUN_GATEWAY_DEPLOY") == "1" and GATEWAY_ID:
    # Targets must be deleted before the gateway.
    for t in gw_admin.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", []):
        gw_admin.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=t["targetId"])
        print("Deleted target:", t["targetId"])
    gw_admin.delete_gateway(gatewayIdentifier=GATEWAY_ID)
    print("Deleted gateway:", GATEWAY_ID)
else:
    print("No gateway created by this notebook - nothing to clean up.")

---

## What you've done

- **Harness engineering (1)** — the deterministic scaffolding that ties the other levers together; a practice with a 7-point production checklist, not a single API.
- **Sub-agent delegation (2)** — a lead stays lean while a cheaper worker does token-heavy work and returns only a summary.
- **GEPA / DSPy (3)** — reflective, eval-driven prompt optimization; conceptual here, run it on your own task later.
- **Tool search via MCP Gateway (4)** — semantic search returns the few relevant tools per query instead of all of them.

These are the levers you reach for **last** — after LOW and MEDIUM are fully applied, measured, and still short of target.